In [1]:
# ViT vs CNN

# Loading ViT model
from transformers import ViTForImageClassification, ViTImageProcessor

model_name = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=10,
)

c:\Personal Projects\ViT_vs_CNN\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
# Loading Dataset
from datasets import load_dataset

dataset = load_dataset("cifar10")
#print(dataset)

# Lets start with only 1000 samples :D
train_data = dataset["train"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"]

print(test_data)
print(train_data)
#print(test_data)

Dataset({
    features: ['img', 'label'],
    num_rows: 10000
})
Dataset({
    features: ['img', 'label'],
    num_rows: 1000
})


In [21]:
# We need to preprocess the images for the ViT model
def preprocess_images(examples):
    inputs = processor(examples['img'])

    inputs['pixel_values'] = [pv for pv in inputs['pixel_values']]
    inputs['labels'] = examples['label']

    return inputs

train_data = train_data.map(preprocess_images, batched=True, remove_columns=train_data.column_names)
test_data = test_data.map(preprocess_images, batched=True, remove_columns=test_data.column_names)

train_data.set_format(type="torch")
test_data.set_format(type="torch")

print(train_data.column_names)
#print(train_data['pixel_values'].shape)

Map: 100%|██████████| 10000/10000 [00:20<00:00, 496.08 examples/s]

['pixel_values', 'labels']


In [22]:
sample = train_data[0]["pixel_values"]
print(sample.shape)

torch.Size([3, 224, 224])


In [ ]:
# Fine-tuning ViT model
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

training_args = TrainingArguments(
    output_dir="./vit-fine-tuned",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    #remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.355857,0.935000
2,0.368300,0.323633,0.922700
3,0.368300,0.250578,0.938400
4,0.111700,0.256782,0.935800
5,0.041600,0.246847,0.937600
6,0.041600,0.247823,0.936900
7,0.027100,0.249623,0.936400
8,0.021300,0.251954,0.935900
9,0.021300,0.253097,0.936000
10,0.018700,0.253528,0.936000


TrainOutput(global_step=320, training_loss=0.0930901356972754, metrics={'train_runtime': 4437.8253, 'train_samples_per_second': 2.253, 'train_steps_per_second': 0.072, 'total_flos': 7.7497545904128e+17, 'train_loss': 0.0930901356972754, 'epoch': 10.0})